# Bizpanion Sales Forecasting LSTM
## Train on Kaggle GPU → Export weights → Load in FastAPI
**Dataset:** [Indian Vegetable Price Dataset](https://www.kaggle.com/datasets/datahack-studio/vegetable-and-fruits-price-in-india)
After training, download `forecast_model.pt` and place in `backend/models/`

In [ ]:
# Install dependencies
!pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
# Kaggle: https://www.kaggle.com/datasets/datahack-studio/vegetable-and-fruits-price-in-india
import os

# Try Kaggle path first
data_paths = [
    '/kaggle/input/vegetable-and-fruits-price-in-india/vegetable_price.csv',
    '/kaggle/input/vegetable-and-fruits-price-in-india/Monthly_data_cmo_2013_16.csv',
]

df = None
for path in data_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded: {path}, shape: {df.shape}')
        break

if df is None:
    # Generate synthetic training data if dataset not found
    print('Dataset not found — generating synthetic training data')
    dates = pd.date_range('2020-01-01', periods=1000, freq='D')
    base_demand = 100
    trend = np.linspace(0, 20, 1000)
    seasonality = 20 * np.sin(2 * np.pi * np.arange(1000) / 365)
    weekly = 10 * np.sin(2 * np.pi * np.arange(1000) / 7)
    noise = np.random.normal(0, 5, 1000)
    demand = base_demand + trend + seasonality + weekly + noise
    demand = np.maximum(demand, 5)
    df = pd.DataFrame({'date': dates, 'quantity': demand})

print(df.head())
print(df.dtypes)

In [ ]:
# ── Preprocess ───────────────────────────────────────────────────────────────
# Handle either raw Agmarknet/CMO format or our synthetic format

if 'quantity' in df.columns:
    series = df['quantity'].values.astype(np.float32)
elif 'Modal Price (Rs./Quintal)' in df.columns:
    # Agmarknet format — use modal price as proxy for demand
    df['date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.sort_values('date').dropna(subset=['date'])
    series = df['Modal Price (Rs./Quintal)'].values.astype(np.float32)
elif 'AVERAGE' in df.columns:
    series = df['AVERAGE'].values.astype(np.float32)
else:
    # Auto-detect numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    series = df[numeric_cols[0]].values.astype(np.float32)

# Clean
series = series[~np.isnan(series)]
series = series[series > 0]
print(f'Series length: {len(series)}, min={series.min():.2f}, max={series.max():.2f}')

# Normalize
scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()
print(f'Scaled range: {series_scaled.min():.3f} to {series_scaled.max():.3f}')

In [ ]:
# ── Create sequences ─────────────────────────────────────────────────────────
SEQ_LEN = 30   # Use 30 days of history
PRED_LEN = 7   # Predict 7 days ahead
BATCH_SIZE = 64

def create_sequences(data, seq_len, pred_len):
    X, y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len:i+seq_len+pred_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = create_sequences(series_scaled, SEQ_LEN, PRED_LEN)
print(f'X shape: {X.shape}, y shape: {y.shape}')

# Train/val split (80/20)
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

X_train_t = torch.tensor(X_train).unsqueeze(-1).to(DEVICE)
y_train_t = torch.tensor(y_train).to(DEVICE)
X_val_t   = torch.tensor(X_val).unsqueeze(-1).to(DEVICE)
y_val_t   = torch.tensor(y_val).to(DEVICE)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

In [ ]:
# ── Model Definition ─────────────────────────────────────────────────────────
class SalesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=7, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers>1 else 0)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = SalesLSTM(input_size=1, hidden_size=64, num_layers=2, output_size=PRED_LEN).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS = 100
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            val_loss += criterion(model(xb), yb).item()
    
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'forecast_model.pt')
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.5f} | Val: {val_loss:.5f}')

print(f'\n✅ Best val loss: {best_val_loss:.5f}')

In [ ]:
# ── Plot training curves ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Bizpanion LSTM — Training Curves')
plt.legend(); plt.tight_layout(); plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Evaluate on val set ──────────────────────────────────────────────────────
model.load_state_dict(torch.load('forecast_model.pt', map_location='cpu'))
model.eval()

with torch.no_grad():
    sample_x = X_val_t[:1]
    pred_scaled = model(sample_x).cpu().numpy()[0]
    actual_scaled = y_val_t[:1].cpu().numpy()[0]

# Inverse transform
pred_actual = scaler.inverse_transform(pred_scaled.reshape(-1,1)).flatten()
actual_actual = scaler.inverse_transform(actual_scaled.reshape(-1,1)).flatten()

mae = np.mean(np.abs(pred_actual - actual_actual))
mape = np.mean(np.abs((pred_actual - actual_actual) / (actual_actual + 1))) * 100

print(f'Sample forecast — MAE: {mae:.2f}, MAPE: {mape:.1f}%')
print(f'Predicted: {pred_actual.round(1)}')
print(f'Actual:    {actual_actual.round(1)}')

In [ ]:
# ── Save model weights ───────────────────────────────────────────────────────
# Download forecast_model.pt from Kaggle output and place in backend/models/
import os
print(f'Model file size: {os.path.getsize("forecast_model.pt") / 1024:.1f} KB')
print('\n✅ DONE! Download forecast_model.pt from Kaggle output panel on the right.')
print('   Place it at: backend/models/forecast_model.pt')